# Homework 6 - Machine Learning Zoomcamp 2026

In this homework, we continue using the fuel efficiency dataset.
The goal of this homework is to create a regression model for predicting the car fuel efficiency (column 'fuel_efficiency_mpg').

In [41]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

## 1.Loading and exploring the data

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-28 19:12:22--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
connected. to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... 
200 OKequest sent, awaiting response... 
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv.3’

car_fuel_efficiency 100%[===================>] 620.29K  3.84MB/s    in 0.2s    

2026-10-28 19:12:23 (3.84 MB/s) - ‘car_fuel_efficiency_2026.csv.3’ saved [635180/635180]



In [3]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
df.shape


(10000, 11)

In [5]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df.describe()

,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
mean,1999.549500,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000
max,2024.000000,5.00000,3110.000000,8.000000,334.000000,5460.000000,23.800000,41.200000


In [7]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

## 2. Prepare and split data

In [8]:
# Fill missing valuse wit 0
df = df.fillna(0)

In [9]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [10]:
# split 
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [11]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [12]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [13]:
dv.get_feature_names_out()

array(['acceleration', 'drivetrain=All-wheel drive',
       'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive',
       'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline',
       'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders',
       'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA',
       'vehicle_weight'], dtype=object)

## Question 1
train a decision tree regressor to predict the fuel_efficiency_mpg variable.

Train a model with max_depth=1.

In [14]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



**Answer: original feature `model_year`**

## Question 2 
Train a random forest regressor with these parameters:

n_estimators=10
random_state=1
n_jobs=-1 (optional - to make training faster)

In [16]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs= -1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
root_mean_squared_error(y_val, y_pred)

1.8393827905033795

**Answer: The RMSE of this model on the validation data is 1.8393827905033795**

## Question 3

Experiment with the n_estimators parameter

* Try these values of this parameter: [10, 50, 100, 150].
* Set random_state to 1.
* Evaluate the model on the validation dataset.

In [18]:
scores  = [] 

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n,
                               random_state=1,
                               n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    
    scores.append((n, round(rmse, 4)))

scores

[(10, 1.8394), (50, 1.7729), (100, 1.7676), (150, 1.7681)]

**Answer: The lowest validation rmse is 150**

## Question 4 

select the best max_depth:

* Try different values of max_depth: [10, 15, 20, 25]
* For each of these values,
* try different values of n_estimators from [10, 50, 100, 150]
* calculate the mean RMSE
* Fix the random seed: random_state=1

In [22]:
scores = [] 

for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, 
                                   max_depth=d, 
                                   random_state=1,
                                   n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        scores.append((d, n, rmse))
scores

[(10, 10, 1.78209618243044),
 (10, 50, 1.7470902075299413),
 (10, 100, 1.7435552997215298),
 (10, 150, 1.7437904052063802),
 (15, 10, 1.829661865928944),
 (15, 50, 1.771177441028383),
 (15, 100, 1.7672297429312616),
 (15, 150, 1.766612105491821),
 (20, 10, 1.840029279522714),
 (20, 50, 1.7744174965456456),
 (20, 100, 1.7691592763835085),
 (20, 150, 1.767779858169317),
 (25, 10, 1.8400106020129927),
 (25, 50, 1.7726834559244742),
 (25, 100, 1.7667317550496153),
 (25, 150, 1.7672723956802336)]

In [25]:
df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
df_scores

,max_depth,n_estimators,rmse
0,10,10,1.782096
1,10,50,1.747090
2,10,100,1.743555
3,10,150,1.743790
4,15,10,1.829662
5,15,50,1.771177
6,15,100,1.767230
7,15,150,1.766612
8,20,10,1.840029
9,20,50,1.774417


In [26]:
df_scores.groupby('max_depth').rmse.mean()


max_depth
10    1.754133
15    1.783670
20    1.787846
25    1.786675
Name: rmse, dtype: float64

**Answer: best max debth 10**

## Question 5:

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the feature_importances_ field.

For this homework question, we'll find the most important feature:

* Train the model with these parameters:
* n_estimators=10,
* max_depth=20,
* random_state=1,
* n_jobs=-1 (optional)

Get the feature importance information from this model

In [30]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_l

In [31]:
rf.feature_importances_

array([0.05534153, 0.00797122, 0.02010786, 0.00557414, 0.06115154,
       0.01804496, 0.16071248, 0.01483121, 0.07846831, 0.29591522,
       0.00722733, 0.01936444, 0.00638824, 0.00356628, 0.04226824,
       0.20306701])

In [35]:
df_importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})

df_importance.sort_values('importance', ascending=False)

,feature,importance
9,model_year,0.295915
15,vehicle_weight,0.203067
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078468
4,engine_displacement,0.061152
0,acceleration,0.055342
14,origin=USA,0.042268
2,drivetrain=Front-wheel drive,0.020108
11,num_doors,0.019364
5,fuel_type=Diesel,0.018045


**Answer: vehicle_weight is the most important feature**

## Question 6:
Train an XGBoost model! For this question, we'll tune the eta parameter:

* Install XGBoost
* Create DMatrix for train and validation
* Create a watchlist
* Train a model with these parameters for 100 rounds:
  
  xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
  }

change eta from 0.3 to 0.1.

In [44]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

In [45]:
evals_result = {}

xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  evals=watchlist, verbose_eval=10,
                  evals_result=evals_result)

rmse_03 = evals_result['val']['rmse']

[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646


In [46]:
evals_result = {}

xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  evals=watchlist, verbose_eval=10,
                  evals_result=evals_result)

rmse_01 = evals_result['val']['rmse']


[0]	train-rmse:2.74739	val-rmse:2.78635
[10]	train-rmse:1.80592	val-rmse:1.93152
[20]	train-rmse:1.55847	val-rmse:1.75076
[30]	train-rmse:1.47544	val-rmse:1.71667
[40]	train-rmse:1.43074	val-rmse:1.71289
[50]	train-rmse:1.40368	val-rmse:1.71358
[60]	train-rmse:1.37884	val-rmse:1.71688
[70]	train-rmse:1.36283	val-rmse:1.71919
[80]	train-rmse:1.34578	val-rmse:1.72100
[90]	train-rmse:1.32546	val-rmse:1.72303
[99]	train-rmse:1.31192	val-rmse:1.72481


In [47]:
print('eta=0.3  final:', rmse_03[-1], ' best:', min(rmse_03))
print('eta=0.1  final:', rmse_01[-1], ' best:', min(rmse_01))


eta=0.3  final: 1.8264579949460302  best: 1.7320478064805087
eta=0.1  final: 1.7248100768778967  best: 1.7121714763059768


**Answer: The eta leading to the best rmse score is 0.1. Eta of 0.1 scored lower in both measures and that is the clear winner.**